# ConformalGuard on R-Judge (Kaggle, results are kept)

Kaggle deletes `/kaggle/working` when an interactive session ends. To keep the results, do **not** run this interactively: set Settings → Internet **on**, choose the accelerator (GPU T4 x2 if you have quota, otherwise None; CPU works, it is only slower), then click **Save Version → Save & Run All (Commit)**. When the run finishes, open the saved version, go to its **Output** tab and download `rjudge_results.zip`.

The zip is rewritten after every finished part, so a run that is cut short still leaves a usable file.

In [ ]:
import os
os.makedirs('/kaggle/temp', exist_ok=True)   # scratch space, not saved as output
%cd /kaggle/temp
!git clone -q --depth 1 -b claude/implement-conformalguard-UGndd https://github.com/rogerpanel/CV.git
!git clone -q --depth 1 https://github.com/Lordog/R-Judge.git
!pip -q install sentence-transformers scikit-learn pytest
%cd /kaggle/temp/CV/ConformalGuard
!git log --oneline -1

In [ ]:
!python -m pytest tests/test_core.py tests/test_adaptive.py tests/test_rjudge.py -q

In [ ]:
def pack():
    # only this zip lands in /kaggle/working, which is what Kaggle keeps
    !cd /kaggle/temp/CV/ConformalGuard && zip -qr /kaggle/working/rjudge_results.zip runs -x '*/cache/*'
    !ls -la /kaggle/working/rjudge_results.zip

## Main run: MiniLM features (the encoder used in the paper's Table 3)

In [ ]:
!python scripts/rjudge_run.py --rjudge-dir ../../R-Judge --features st \
    --st-model sentence-transformers/all-MiniLM-L6-v2 \
    --reps 100 --loco-reps 10 --out runs/rjudge_st_minilm
pack()

## Second encoder: BGE-base (appendix)

In [ ]:
!python scripts/rjudge_run.py --rjudge-dir ../../R-Judge --features st \
    --st-model BAAI/bge-base-en-v1.5 \
    --reps 100 --loco-reps 10 --out runs/rjudge_st_bge
pack()